Path setup

In [ ]:
import sys
sys.path.append("..")

Imports

In [ ]:
from src.similarity.vectorizer import TfidfSimilarityVectorizer
from src.similarity.similarity_scorer import SimilarityScorer
from src.similarity.batch_comparator import compare_students_to_master

Test the vectorizer in isolation

In [ ]:
vectorizer = TfidfSimilarityVectorizer()

docs = [
    "the cell membrane controls what enters and exits the cell",
    "the cell membrane regulates substances entering and leaving the cell",
    "photosynthesis converts light energy into chemical energy in plants"
]

matrix = vectorizer.fit_transform(docs)
print("Matrix shape:", matrix.shape)
print("Vocabulary size:", len(vectorizer.vectorizer.vocabulary_))

Test the similarity scorer in isolation

In [ ]:
scorer = SimilarityScorer()

# doc 0 and doc 1 are paraphrases (should score high)
score_similar = scorer.compute_score(matrix[0], matrix[1])
print("Similar docs score:", score_similar, "->", scorer.match_level(score_similar))

# doc 0 and doc 2 are unrelated (should score low)
score_different = scorer.compute_score(matrix[0], matrix[2])
print("Different docs score:", score_different, "->", scorer.match_level(score_different))

Sanity check

In [ ]:
identical_docs = ["this is a test sentence", "this is a test sentence"]
identical_matrix = vectorizer.fit_transform(identical_docs)
score_identical = scorer.compute_score(identical_matrix[0], identical_matrix[1])
print("Identical text score:", score_identical)

Full pipeline test with real sample files

In [ ]:
student_files = ["../data/samples/ANS_DOCX.docx", "../data/samples/ANS_TXT.txt"]
master_file = "../data/samples/ANS_PDF.pdf"  # deliberately reuse one file as "student" too

df = compare_students_to_master(student_files, master_file)
df

Deliberate failure test

In [ ]:
student_files_with_bad_file = [
    "../data/samples/ANS_PDF.pdf",
    "../data/samples/does_not_exist.pdf"
]
df_with_error = compare_students_to_master(student_files_with_bad_file, master_file)
df_with_error

Inspect score distribution across all samples

In [ ]:
import os

all_files = [os.path.join("../data/samples", f) for f in os.listdir("../data/samples")]
df_all = compare_students_to_master(all_files, master_file)
df_all.sort_values("similarity_score", ascending=False)